# E3.2b — Matched-Cutoff Coordination (standalone)

The one trajectory-touching unit of the cross-solvent phase, split out for
provenance. Per-system coordination used pair-specific first-shell cutoffs, so
magnitudes are not comparable across solvents; this recomputes each resolved
(reference, partner) at a **common** cutoff per pair-type — derived from the
committed `rdf_cutoffs.csv` (`first_min_nm`), not invented — using the same
estimator as the per-system phase (count of partner atoms within the cutoff,
ACF-corrected CI).

**I/O.** Inputs: committed coordination/rdf CSVs (re-loaded here, not inherited
from the reduction notebook) plus the 35 production DCDs. Output: the guarded
cache `e3.2_matched_cutoff.csv`, which `E3x_cross_solvent.ipynb` §2a/§4 consume.

**Cost.** ~4 min first run (loads dominate: 35 strided DCD loads at ~5 s each;
RDF/count compute is a minor term). Guarded by the cache so it runs exactly once;
delete `e3.2_matched_cutoff.csv` to force a recompute.

**Provenance guarantee.** Reference atom sets are reconstructed from the peptide
sequences and motif spans and **validated against the committed `n_ref`** (§1
pre-flight) before any trajectory is read — so the recompute is provably the
same atom sets as the per-system phase, and the §4 QC re-confirms it by
reproducing stored `pooled_coord` at common cutoffs.

## §0 — Configuration and committed inputs

Re-derives `data` (coordination + rdf CSVs), `COL`, and the `long` resolved-shell
frame from committed outputs, so this notebook is self-contained. `CS_MODULE_DIR`
points at `extension/` where `convergence_stats.py` lives (distinct from the
convergence sidecar CSVs under `extension/analysis/convergence/`).

In [1]:
import os, sys, time
import numpy as np
import pandas as pd
import mdtraj as md

PROJECT   = os.path.expanduser("~/des-peptide-study")
E3X       = os.path.join(PROJECT, "extension/analysis/E3x")
TRAJROOT  = os.path.join(PROJECT, "extension/trajectories_extended")
SYSTOP    = os.path.join(PROJECT, "systems")
OUTDIR    = os.path.join(E3X, "cross_solvent")
CS_MODULE_DIR = os.path.join(PROJECT, "extension")   # convergence_stats.py lives here
os.makedirs(OUTDIR, exist_ok=True)

PEPTIDES = ["GGE", "CME", "YIY"]
SOLVENTS = ["water", "reline", "glyceline"]
SYSTEMS  = [f"{p}_{s}" for p in PEPTIDES for s in SOLVENTS]
STARTS   = ["compact", "mid", "open", "extended"]

STRIDE          = 10
WIN_START_FRAME = 2000        # drop 0-20 ns at stride-10 -> 18,000 frames

def traj_path(system, start):
    tag = f"{system}_200ns_{start}_r1"
    return os.path.join(TRAJROOT, tag, tag + ".dcd")

def top_path(system):
    return os.path.join(SYSTOP, system, system + ".prmtop")

CACHE = os.path.join(OUTDIR, "e3.2_matched_cutoff.csv")
print("outputs ->", OUTDIR)
print("cache   ->", CACHE, "(exists)" if os.path.exists(CACHE) else "(absent - will recompute)")

outputs -> /Users/rossgibson/des-peptide-study/extension/analysis/E3x/cross_solvent
cache   -> /Users/rossgibson/des-peptide-study/extension/analysis/E3x/cross_solvent/e3.2_matched_cutoff.csv (absent - will recompute)


In [2]:
# committed coordination + rdf inputs, and the COL map (reconciled schema)
COL = {
    "site": "reference", "partner": "partner",
    "cutoff_nm": "cutoff_nm", "coord": "pooled_coord",
    "coord_ci95": "ci95", "resolved": "structured_shell", "per_atom": "per_atom",
    "first_min": "first_min_nm",
}
SECTION = {
    "backbone":    "coordination/backbone_coordination.csv",
    "sidechain":   "coordination/sidechain_coordination.csv",
    "rdf_cutoffs": "rdfs/rdf_cutoffs.csv",
}
def _read(system, rel):
    p = os.path.join(E3X, system, rel)
    return pd.read_csv(p) if os.path.isfile(p) else None

data = {s: {k: _read(s, rel) for k, rel in SECTION.items()} for s in SYSTEMS}
missing = [f"{s}:{k}" for s in SYSTEMS for k in SECTION if data[s][k] is None]
assert not missing, f"missing committed inputs: {missing}"
print("loaded coordination + rdf inputs for all", len(SYSTEMS), "systems")

loaded coordination + rdf inputs for all 9 systems


In [3]:
# rebuild the resolved-shell `long` frame (structured_shell == True only)
rows = []
for s in SYSTEMS:
    pep, solv = s.split("_")[0], s.split("_", 1)[1]
    for scope in ("backbone", "sidechain"):
        df = data[s][scope]
        for _, r in df.iterrows():
            if not bool(r[COL["resolved"]]):
                continue
            rows.append({"peptide": pep, "solvent": solv, "scope": scope,
                         "site": str(r[COL["site"]]), "partner": str(r[COL["partner"]]),
                         "cutoff_nm": r.get(COL["cutoff_nm"], np.nan),
                         "coordination": r.get(COL["coord"], np.nan),
                         "per_atom": r.get(COL["per_atom"], np.nan)})
long = pd.DataFrame(rows)
print(f"resolved shells: {len(long)} rows across {long.site.nunique()} references, "
      f"{long.partner.nunique()} partners")
print(sorted(long.site.unique()), "|", sorted(long.partner.unique()))

resolved shells: 57 rows across 6 references, 6 partners
['backbone_O', 'flank_Glu', 'motif_Cys', 'motif_Glu', 'motif_Met', 'motif_Tyr'] | ['chloride', 'choline_N', 'glycerol_O', 'urea_N', 'urea_O', 'water_O']


## §1 — Reference/partner resolver, validated against committed n_ref

Reference atom sets are reconstructed from the motif spans (topology resSeq,
`ACE=0`): GGE and CME motifs at residues 4–6, YIY at 3–5. This separates GGE's
single `motif_Glu` (E6) from the `flank_Glu` cluster (E1–E3), which a resname
string cannot. Partner atoms use the real topology names surfaced from the
prmtops. The pre-flight cell asserts each reconstructed reference count equals
the committed `n_ref` (= `pooled_coord / per_atom`) **before** any trajectory is
read — the gate that makes the recompute provably the per-system atom sets.

In [5]:
# motif span (resSeq, ACE=0) — the 3-residue motif tripeptide per peptide
MOTIF = {"GGE": (4, 5, 6), "CME": (4, 5, 6), "YIY": (3, 4, 5)}

def ref_spec(peptide):
    m = set(MOTIF[peptide])
    spec = {"backbone_O": (m, {"O"})}                 # motif carbonyl O (3 atoms)
    if peptide == "GGE":
        spec["motif_Glu"] = ({6},       {"OE1", "OE2"})   # single motif Glu (E6)
        spec["flank_Glu"] = ({1, 2, 3}, {"OE1", "OE2"})   # EEE cluster
    elif peptide == "CME":
        spec["motif_Cys"] = ({4}, {"SG"})
        spec["motif_Met"] = ({5}, {"SD"})
        spec["motif_Glu"] = ({6}, {"OE1", "OE2"})
    elif peptide == "YIY":
        spec["motif_Tyr"] = ({3, 5}, {"OH"})
    return spec

# partner label -> (residue name, atom-name set | None for single-atom residue)
PARTNER = {
    "water_O":    ("HOH", {"O"}),
    "urea_O":     ("URE", {"O"}),
    "urea_N":     ("URE", {"N", "N1"}),
    "glycerol_O": ("GOL", {"O", "O1", "O2"}),
    "choline_N":  ("CHO", {"N"}),
    "chloride":   ("CLA", None),
}

def ref_idx(top, peptide, label):
    resset, names = ref_spec(peptide)[label]
    return [a.index for a in top.atoms
            if a.residue.is_protein and a.residue.resSeq in resset and a.name in names]

def partner_idx(top, label):
    rn, names = PARTNER[label]
    return [a.index for a in top.atoms
            if a.residue.name == rn and (names is None or a.name in names)]

In [6]:
# pre-flight gate: reconstructed n_ref must equal committed (coordination / per_atom)
tops = {p: md.load_prmtop(top_path(f"{p}_water")) for p in PEPTIDES}   # site count is solvent-independent

stored = (long.assign(nref=(long["coordination"] / long["per_atom"]).round())
              .dropna(subset=["nref"])
              .groupby(["peptide", "site"])["nref"].first())

print(f"{'peptide':8s} {'site':12s} {'recon':>5s} {'stored':>6s}")
ok = True
for (pep, site), nref_stored in stored.items():
    if site not in ref_spec(pep):
        print(f"{pep:8s} {site:12s} {'--':>5s} {int(nref_stored):>6d}  <-- no ref_spec"); ok = False; continue
    n_recon = len(ref_idx(tops[pep], pep, site))
    flag = "" if n_recon == int(nref_stored) else "  <-- MISMATCH"
    print(f"{pep:8s} {site:12s} {n_recon:>5d} {int(nref_stored):>6d}{flag}")
    ok = ok and (n_recon == int(nref_stored))
assert ok, "reference reconstruction disagrees with committed n_ref — fix resolver before recompute"
print("\nall reference atom-sets validated against committed n_ref")

# partner presence check on one DES topology (labels resolve to >0 atoms)
_t = md.load_prmtop(top_path("YIY_glyceline"))
for lab in ("glycerol_O", "choline_N", "chloride", "water_O"):
    print(f"  {lab:11s} -> {len(partner_idx(_t, lab))} atoms")
_t = md.load_prmtop(top_path("GGE_reline"))
for lab in ("urea_O", "urea_N", "choline_N", "chloride"):
    print(f"  {lab:11s} -> {len(partner_idx(_t, lab))} atoms")

peptide  site         recon stored
CME      backbone_O       3      3
CME      motif_Cys        1      1
CME      motif_Glu        2      2
CME      motif_Met        1      1
GGE      backbone_O       3      3
GGE      flank_Glu        6      6
GGE      motif_Glu        2      2
YIY      backbone_O       3      3
YIY      motif_Tyr        2      2

all reference atom-sets validated against committed n_ref
  glycerol_O  -> 60 atoms
  choline_N   -> 10 atoms
  chloride    -> 10 atoms
  water_O     -> 751 atoms
  urea_O      -> 26 atoms
  urea_N      -> 52 atoms
  choline_N   -> 13 atoms
  chloride    -> 13 atoms


## §2 — Matched cutoff per pair-type

Common cutoff per (reference, partner) from the committed `first_min_nm`, taking
the `max` across the solvents that resolve it (captures the full first shell
everywhere). NaN entries (no shell) are dropped. Recompute set = every resolved
(site, partner) in the matrix, augmented with `(site, water_O)` across all three
solvents for every site that resolves a water shell — the hydration-across-
solvents comparison that is the core indirect-restructuring signal.

In [7]:
MATCHED_RULE = "max"   # {"max","mean","min"}
cut_rows = []
for s in SYSTEMS:
    rc = data[s]["rdf_cutoffs"]
    for _, r in rc.iterrows():
        v = r[COL["first_min"]]
        if pd.isna(v):
            continue
        cut_rows.append({"site": str(r[COL["site"]]), "partner": str(r[COL["partner"]]),
                         "solvent": s.split("_", 1)[1], "cutoff_nm": float(v)})
matched_cut = (pd.DataFrame(cut_rows).groupby(["site", "partner"])["cutoff_nm"]
                 .agg(MATCHED_RULE).rename("matched_cutoff_nm").reset_index())
print("matched cutoffs per pair-type:")
print(matched_cut.to_string(index=False))

want = matched_cut.merge(
    long[["peptide", "solvent", "site", "partner"]].drop_duplicates(),
    on=["site", "partner"], how="inner")

aug = []
for p in PEPTIDES:
    sites_p = long.loc[(long.peptide == p) & (long.partner == "water_O"), "site"].unique()
    for st in sites_p:
        for sv in SOLVENTS:
            aug.append({"site": st, "partner": "water_O", "peptide": p, "solvent": sv})
aug = pd.DataFrame(aug).merge(matched_cut, on=["site", "partner"], how="left")
want = (pd.concat([want, aug], ignore_index=True)
          .dropna(subset=["matched_cutoff_nm"])
          .drop_duplicates(["peptide", "solvent", "site", "partner"]))
print(f"\n{len(want)} (system-pair) recomputes across {want.peptide.nunique()} peptides")

matched cutoffs per pair-type:
      site    partner  matched_cutoff_nm
backbone_O  choline_N              0.423
backbone_O glycerol_O              0.322
backbone_O     urea_N              0.423
backbone_O     urea_O              0.562
backbone_O    water_O              0.338
 flank_Glu  choline_N              0.527
 flank_Glu glycerol_O              0.317
 flank_Glu     urea_N              0.392
 flank_Glu    water_O              0.327
 motif_Cys  choline_N              0.442
 motif_Cys glycerol_O              0.412
 motif_Cys     urea_N              0.447
 motif_Cys     urea_O              0.377
 motif_Glu  choline_N              0.548
 motif_Glu glycerol_O              0.317
 motif_Glu     urea_N              0.392
 motif_Glu    water_O              0.327
 motif_Met  choline_N              0.447
 motif_Met     urea_N              0.433
 motif_Tyr   chloride              0.382
 motif_Tyr glycerol_O              0.538
 motif_Tyr     urea_N              0.387
 motif_Tyr     urea_O     

## §3 — Recompute (guarded, cached, one pass)

One system at a time: load its starts once, count partner atoms within the
matched cutoff per resolved reference, ACF-correct via `compute_stat`, pool
across starts. Writes `e3.2_matched_cutoff.csv`. Skips entirely if the cache
exists.

In [8]:
if os.path.exists(CACHE):
    matched = pd.read_csv(CACHE)
    print(f"cache present ({len(matched)} rows) — skipping recompute; delete {os.path.basename(CACHE)} to force")
else:
    sys.path.insert(0, CS_MODULE_DIR)
    try:
        import convergence_stats as cs      # compute_stat(series, dt_ps=10.0) -> dict
        HAVE_CS = True
    except Exception as e:
        HAVE_CS = False
        print(f"[check] convergence_stats import failed ({e}); raw CI fallback (understates)")

    def _count_within(traj, a_idx, b_idx, cutoff, chunk=500):
        if not a_idx or not b_idx:
            return None, 0
        pairs = np.array([[i, j] for i in a_idx for j in b_idx])
        counts = []
        for k in range(0, traj.n_frames, chunk):
            d = md.compute_distances(traj[k:k+chunk], pairs)
            counts.extend((d < cutoff).sum(axis=1).tolist())
        return np.asarray(counts, float), len(a_idx)

    out_rows = []
    for s in SYSTEMS:
        pep, solv = s.split("_")[0], s.split("_", 1)[1]
        pairs_here = want[(want.peptide == pep) & (want.solvent == solv)]
        if pairs_here.empty:
            continue
        t0 = time.perf_counter()
        for start in STARTS:
            p = traj_path(s, start)
            if not os.path.isfile(p):
                print(f"  [skip] {s} {start}: not found"); continue
            tr = md.load(p, top=top_path(s), stride=STRIDE)[WIN_START_FRAME:]
            for _, pr in pairs_here.iterrows():
                a_idx = ref_idx(tr.topology, pep, pr.site)
                b_idx = partner_idx(tr.topology, pr.partner)
                series, n_ref = _count_within(tr, a_idx, b_idx, pr.matched_cutoff_nm)
                if series is None:
                    print(f"  [check] empty selection {s} {pr.site}/{pr.partner}"); continue
                if HAVE_CS:
                    st = cs.compute_stat(series, dt_ps=float(STRIDE))
                    m, ci, tau, neff = st["mean"], st["ci95"], st["tau_int_ns"], st["N_eff"]
                else:
                    m, ci = float(series.mean()), 1.96*series.std()/np.sqrt(len(series))
                    tau, neff = np.nan, np.nan
                out_rows.append({"system": s, "peptide": pep, "solvent": solv,
                                 "site": pr.site, "partner": pr.partner,
                                 "matched_cutoff_nm": pr.matched_cutoff_nm, "start": start,
                                 "coord": m, "per_atom": m/max(n_ref, 1),
                                 "ci95": ci, "tau_int_ns": tau, "N_eff": neff})
        print(f"  {s}: {len(pairs_here)} pairs x starts in {time.perf_counter()-t0:.1f}s")

    per_start = pd.DataFrame(out_rows)
    matched = (per_start.groupby(["system", "peptide", "solvent", "site", "partner", "matched_cutoff_nm"])
               .agg(coord=("coord", "mean"), per_atom=("per_atom", "mean"),
                    ci95=("ci95", lambda v: float(np.sqrt(np.sum(np.square(v)))/len(v))),
                    tau_int_ns=("tau_int_ns", "mean"), N_eff=("N_eff", "mean"),
                    n_starts=("start", "nunique"))
               .reset_index())
    matched.to_csv(CACHE, index=False)
    print(f"\nwrote {CACHE} ({len(matched)} pooled rows)")

print(matched.head(24).to_string(index=False))

  GGE_water: 3 pairs x starts in 26.2s
  [skip] GGE_reline open: not found
  GGE_reline: 10 pairs x starts in 28.5s
  GGE_glyceline: 8 pairs x starts in 34.6s
  CME_water: 2 pairs x starts in 23.9s
  CME_reline: 10 pairs x starts in 37.6s
  CME_glyceline: 8 pairs x starts in 34.5s
  YIY_water: 2 pairs x starts in 24.1s
  YIY_reline: 8 pairs x starts in 29.5s
  YIY_glyceline: 6 pairs x starts in 30.9s

wrote /Users/rossgibson/des-peptide-study/extension/analysis/E3x/cross_solvent/e3.2_matched_cutoff.csv (57 pooled rows)
       system peptide   solvent       site    partner  matched_cutoff_nm    coord  per_atom     ci95  tau_int_ns       N_eff  n_starts
CME_glyceline     CME glyceline backbone_O  choline_N              0.423 0.186236  0.062079 0.040136    1.180771  442.580021         4
CME_glyceline     CME glyceline backbone_O    water_O              0.338 2.697778  0.899259 0.299643    5.862232  231.072243         4
CME_glyceline     CME glyceline  motif_Cys  choline_N              0.4

## §4 — QC: recompute reproduces stored coordination at common cutoffs

Where the matched cutoff equals a stored pair-specific cutoff, the recompute
must reproduce that stored `pooled_coord` within CI — the check that the
reconstructed atom sets and estimator match the per-system phase. Differences on
differing-cutoff pairs are expected (that is the point of matching); differences
on common-cutoff pairs flag estimator or selection drift and must be inspected
before the magnitudes enter the synthesis.

In [9]:
qc = matched.merge(long, on=["peptide", "solvent", "site", "partner"], how="inner",
                   suffixes=("_matched", "_stored"))
if len(qc):
    qc["same_cutoff"] = np.isclose(qc["matched_cutoff_nm"], qc["cutoff_nm"], atol=1e-3)
    qc["abs_diff"] = (qc["coord"] - qc["coordination"]).abs()
    common = qc[qc["same_cutoff"]].sort_values("abs_diff", ascending=False)
    print("common-cutoff pairs (should reproduce stored pooled_coord within CI):")
    print(common[["system", "site", "partner", "matched_cutoff_nm",
                  "coord", "coordination", "abs_diff"]].to_string(index=False))
    worst = common["abs_diff"].max() if len(common) else 0.0
    print(f"\nmax abs_diff on common-cutoff pairs: {worst:.4f}")
    print("PASS: reconstruction reproduces per-system estimator" if worst < 0.15
          else "INSPECT: common-cutoff drift exceeds tolerance")
else:
    print("[check] no overlap between recompute and stored shells — inspect keys")

common-cutoff pairs (should reproduce stored pooled_coord within CI):
       system       site    partner  matched_cutoff_nm     coord  coordination  abs_diff
YIY_glyceline  motif_Tyr    water_O              0.358  5.455250        5.4553  0.000050
YIY_glyceline  motif_Tyr   chloride              0.382  0.056944        0.0569  0.000044
   CME_reline  motif_Glu    water_O              0.327  5.225042        5.2250  0.000042
   CME_reline  motif_Met  choline_N              0.447  0.124542        0.1245  0.000042
   YIY_reline  motif_Tyr     urea_N              0.387  0.554639        0.5546  0.000039
GGE_glyceline  flank_Glu    water_O              0.327 16.760139       16.7601  0.000039
   YIY_reline backbone_O    water_O              0.338  4.270236        4.2702  0.000036
   YIY_reline  motif_Tyr     urea_O              0.312  0.099764        0.0998  0.000036
    CME_water  motif_Glu    water_O              0.327  5.129333        5.1293  0.000033
   CME_reline  motif_Met     urea_N     

## Consume in the synthesis

With `e3.2_matched_cutoff.csv` written and QC passed, re-run §4 of
`E3x_cross_solvent.ipynb` to resolve the E3.2 line from `PENDING` to the
direct-vs-indirect adjudication — including the backbone-hydration-across-
solvents comparison (`(backbone_O, water_O)` per solvent) that tests whether
water displacement is reline-specific or common to both DES. Commit this
notebook, its builder, and the cache as the E3.2b unit.

In [10]:
import os, numpy as np, pandas as pd

OUT = os.path.expanduser("~/des-peptide-study/extension/analysis/E3x/cross_solvent")
matched = pd.read_csv(os.path.join(OUT, "e3.2_matched_cutoff.csv"))
P = os.path.join(OUT, "E3x_cross_solvent_synthesis.md")

def q(pep, solv, site, partner, col):
    r = matched[(matched.peptide==pep)&(matched.solvent==solv)&
                (matched.site==site)&(matched.partner==partner)]
    return None if r.empty else float(r.iloc[0][col])

def pa(pep, solv, site, partner):            # per-atom value + per-atom CI
    per = q(pep,solv,site,partner,"per_atom"); coord = q(pep,solv,site,partner,"coord")
    ci  = q(pep,solv,site,partner,"ci95")
    ci_pa = None if None in (per,coord,ci) or coord==0 else ci*per/coord
    return per, ci_pa

def diff(m1,c1,m2,c2):
    if None in (m1,c1,m2,c2): return None,None,None
    d=m1-m2; ci=float(np.hypot(c1,c2)); return d,ci,abs(d)>ci

hw,cw = pa("YIY","water","backbone_O","water_O")
hr,cr = pa("YIY","reline","backbone_O","water_O")
hg,cg = pa("YIY","glyceline","backbone_O","water_O")
drg,cirg,res_rg = diff(hr,cr,hg,cg)
rg_clause = ("resolves a small difference" if res_rg else
             "does not resolve a difference (statistically equal displacement)")

gge_uo,_ = pa("GGE","reline","backbone_O","urea_O"); gge_un,_ = pa("GGE","reline","backbone_O","urea_N")
yiy_uo,_ = pa("YIY","reline","backbone_O","urea_O"); yiy_go,_ = pa("YIY","glyceline","backbone_O","glycerol_O")
cme_cn = q("CME","reline","motif_Cys","urea_N","per_atom"); cme_co = q("CME","reline","motif_Cys","urea_O","per_atom")
cme_mn = q("CME","reline","motif_Met","urea_N","per_atom")

L = ["## E3.2 — direct contact versus indirect restructuring\n"]
L.append(f"- Backbone hydration is displaced by both deep eutectics, not reline alone, and by "
         f"essentially equal amounts. At matched cutoff on the three-atom YIY motif backbone "
         f"(waters per carbonyl O): water {hw:.2f}, reline {hr:.2f}, glyceline {hg:.2f}; the "
         f"reline−glyceline difference {drg:+.3f} [±{cirg:.3f}] {rg_clause}. Backbone dehydration "
         f"is a shared DES property. [REF: e3.2_matched_cutoff.csv]")
L.append(f"- This decouples backbone dehydration from the SASA effect: reline opens YIY SASA "
         f"(+0.135, excludes zero) while glyceline does not (+0.002, includes zero), yet both "
         f"displace backbone water equally — the SASA opening is not explained by backbone water "
         f"displacement. [REF]")
L.append(f"- What distinguishes reline is direct urea occupancy of the vacated sites, not the "
         f"vacancy. Urea contacts the motif backbone in GGE_reline (urea-O {gge_uo:.3f}, urea-N "
         f"{gge_un:.3f} per O) and YIY_reline (urea-O {yiy_uo:.3f}); glycerol reaches the YIY "
         f"backbone only weakly ({yiy_go:.3f}) and the GGE backbone not at all. Reline replaces "
         f"backbone water with urea where glyceline leaves it unreplaced. [REF]")
L.append(f"- The contact mode is not uniform, so no single mechanism accounts for the SASA opening. "
         f"CME_reline opens SASA (+0.248, excludes zero) with no urea–backbone shell — urea instead "
         f"occupies the CME side chains (Cys urea-N {cme_cn:.3f}, urea-O {cme_co:.3f}; Met urea-N "
         f"{cme_mn:.3f}). Direct urea occupancy is the reline-specific feature, but its location "
         f"(backbone in GGE/YIY, side chain in CME) differs by peptide. [REF]")
L.append(f"- Read together these are mechanism-constraining observations, not an established "
         f"mechanism: backbone dehydration is shared and does not track SASA opening; reline's "
         f"distinguishing feature is direct urea occupancy; the occupancy site differs by peptide. "
         f"Choline coordinates backbone and side chains pervasively in both DES (§2a) and is not a "
         f"discriminating variable. Equilibrium and causal status are unresolved (tier caveats on "
         f"GGE_water and the multi-basin systems stand); a causal test of urea occupancy against "
         f"SASA is deferred (Paper 2). [REF]")
e32 = "\n".join(L)

txt = open(P, encoding="utf-8").read()
assert "## E3.2" in txt and "## E3.3" in txt, "E3.2/E3.3 headers not both found"
head = txt.split("## E3.2")[0].rstrip() + "\n\n"
tail = "## E3.3" + txt.split("## E3.3", 1)[1]
open(P, "w", encoding="utf-8").write(head + e32.rstrip() + "\n\n" + tail)
print("E3.2 rewritten; reline-glyceline resolved:", res_rg, "->", rg_clause)
print(e32)

E3.2 rewritten; reline-glyceline resolved: False -> does not resolve a difference (statistically equal displacement)
## E3.2 — direct contact versus indirect restructuring

- Backbone hydration is displaced by both deep eutectics, not reline alone, and by essentially equal amounts. At matched cutoff on the three-atom YIY motif backbone (waters per carbonyl O): water 1.80, reline 1.42, glyceline 1.41; the reline−glyceline difference +0.014 [±0.025] does not resolve a difference (statistically equal displacement). Backbone dehydration is a shared DES property. [REF: e3.2_matched_cutoff.csv]
- This decouples backbone dehydration from the SASA effect: reline opens YIY SASA (+0.135, excludes zero) while glyceline does not (+0.002, includes zero), yet both displace backbone water equally — the SASA opening is not explained by backbone water displacement. [REF]
- What distinguishes reline is direct urea occupancy of the vacated sites, not the vacancy. Urea contacts the motif backbone in GGE_r